In [1]:
import re
import numpy as np
import pandas as pd
from rapidfuzz import fuzz

schools = pd.read_csv("../data/processed/schools_with_athletics.csv")
sevp = pd.read_csv("../data/raw/sevp/sevp_certified_schools.csv")
sevp = sevp[sevp["f_visa"] == "Y"].copy()   # only F-1 certification matters for degree students
print("Schools:", schools.shape, "| SEVP F-1 certified rows:", sevp.shape)

Schools: (3147, 44) | SEVP F-1 certified rows: (13267, 8)


In [2]:
# Normalize names so trivial differences (case, punctuation, abbreviations) don't block a match
ABBREV = {
    r"\bcoll\b": "college",
    r"\buniv\b": "university",
    r"\bcc\b": "community college",
    r"\bcomm\b": "community",
    r"\bctr\b": "center",
    r"\bmt\b": "mount",
}   # NOTE: "st" is left alone on purpose -- it can mean "Saint" OR "State"

def norm(s):
    s = str(s).lower().replace("&", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)              # punctuation -> space
    for pat, rep in ABBREV.items():
        s = re.sub(pat, rep, s)
    s = re.sub(r"\b(the|inc|llc)\b", " ", s)       # filler words
    s = re.sub(r"\bmain campus\b", " ", s)         # Scorecard often appends "-Main Campus"
    return re.sub(r"\s+", " ", s).strip()

schools["name_n"] = schools["name"].apply(norm)
schools["city_n"] = schools["city"].astype(str).str.lower().str.strip()
sevp["school_n"] = sevp["school_name"].apply(norm)
sevp["campus_n"] = sevp["campus_name"].apply(norm)
sevp["city_n"] = sevp["city"].astype(str).str.lower().str.strip()
print("Example:", schools["name"].iloc[0], "->", schools["name_n"].iloc[0])

Example: Alabama A & M University -> alabama a and m university


In [3]:
sevp_by_state = {st: g for st, g in sevp.groupby("state")}   # blocking: compare within state only

def link(row):
    cand = sevp_by_state.get(row["state"])
    if cand is None:
        return pd.Series([None, None, 0.0])
    # Stage 1: exact normalized match on school OR campus name
    exact = cand[(cand["school_n"] == row["name_n"]) | (cand["campus_n"] == row["name_n"])]
    if len(exact):
        return pd.Series(["exact", exact.iloc[0]["campus_name"], 100.0])
    # Stage 2: fuzzy match (token_sort_ratio: ignores word order, penalizes extra words)
    s1 = cand["school_n"].apply(lambda x: fuzz.token_sort_ratio(row["name_n"], x))
    s2 = cand["campus_n"].apply(lambda x: fuzz.token_sort_ratio(row["name_n"], x))
    score = np.maximum(s1, s2)
    best = score.idxmax()
    sc = float(score[best])
    same_city = cand.loc[best, "city_n"] == row["city_n"]
    method = "fuzzy" if (sc >= 92 or (sc >= 85 and same_city)) else None
    # Keep the best candidate even when rejected, so misses can be reviewed
    return pd.Series([method, cand.loc[best, "campus_name"], round(sc, 1)])

schools[["sevp_match", "sevp_best_name", "sevp_score"]] = schools.apply(link, axis=1)
schools["sevp_certified"] = schools["sevp_match"].notna()

print("Match method:")
print(schools["sevp_match"].value_counts(dropna=False).to_string())
print("\nCertified by school type:")
print(pd.crosstab(schools["school_type"], schools["sevp_certified"], margins=True))
print("\n20 random FUZZY matches (review: are these the same school?):")
fz = schools[schools["sevp_match"] == "fuzzy"]
print(fz.sample(min(20, len(fz)), random_state=1)[["name", "state", "sevp_best_name", "sevp_score"]].to_string(index=False))

Match method:
sevp_match
exact    2247
None      774
fuzzy     123
NaN         3

Certified by school type:
sevp_certified  False  True   All
school_type                      
2-year            493   857  1350
4-year            284  1513  1797
All               777  2370  3147

20 random FUZZY matches (review: are these the same school?):
                                           name state                           sevp_best_name  sevp_score
   Eastern New Mexico University-Roswell Campus    NM    Eastern New Mexico University-Roswell        91.4
         Yeshiva College of the Nations Capital    MD  Yeshiva College of the Nation's Capital        95.7
                 St. John's University-New York    NY                                 Brooklyn        93.3
 Inter American University of Puerto Rico-Ponce    PR                           Fajardo Campus        93.0
Rowan College of South Jersey-Cumberland Campus    NJ Rowan College of South Jersey-Cumberland        92.0
                C

In [4]:
# Error detector: schools with 2%+ international students can't have them without SEVP certification
# (mostly). Unmatched ones are probably linkage misses, not truly uncertified schools.
miss = schools[(~schools["sevp_certified"]) & (schools["pct_international"] >= 0.02)]
print("Likely linkage misses (2%+ international but not matched):", len(miss))
print(miss.nlargest(25, "pct_international")[
    ["name", "state", "pct_international", "sevp_best_name", "sevp_score"]
].round(3).to_string(index=False))

print("\nKansas 2-year schools:")
print(schools[(schools["state"] == "KS") & (schools["school_type"] == "2-year")][
    ["name", "sevp_certified", "sevp_match", "sevp_best_name", "sevp_score"]
].sort_values("name").to_string(index=False))

schools.to_csv("../data/processed/schools_with_sevp.csv", index=False)
print("\nSaved:", schools.shape, "-> data/processed/schools_with_sevp.csv")

Likely linkage misses (2%+ international but not matched): 132
                                                 name state  pct_international                           sevp_best_name  sevp_score
                The New England Conservatory of Music    MA              0.433                 New England Conservatory        84.2
                                Saint John's Seminary    MA              0.429                      St. John's Seminary        76.9
                    Lindsey Hopkins Technical College    FL              0.244               Southern Technical College        67.8
   Trine University-Regional/Non-Traditional Campuses    IN              0.236  Trine University-Ft. Wayne Regional Ctr        63.7
                  Yeshiva Gedolah of Woodlake Village    NJ              0.218                            Ohr Zechariah        66.7
                             American Islamic College    IL              0.188                   Christian Life College        65.2
             